In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
text = ['hello, good morning' ]

In [39]:
vect = TfidfVectorizer()

In [40]:
vect.fit(text)

TfidfVectorizer()

In [41]:
## TF will count the frequency of word in each document. and IDF 
print(vect.idf_)

[1. 1. 1.]


In [42]:
print(vect.vocabulary_)

{'hello': 1, 'good': 0, 'morning': 2}


In [43]:
example = text[0]
example

'hello, good morning'

In [44]:
example = vect.transform([example])
print(example.toarray())

[[0.57735027 0.57735027 0.57735027]]


In [2]:
import pandas as pd

In [3]:
dataframe = pd.read_csv('cleaned_dataset.csv')
dataframe.head()

,id,statement,description,rating
0,6771.0,Dawn dish soap contains ammonia even though it...,Could Mixing Dawn Dish Soap with Clorox Bleach...,mixture
1,2468.0,Is blue light harmful to our eyes?,It seems reasonable to reduce exposure to blue...,unproven
2,1929.0,Are the 'Winter Blues' real?,"I've recently <a href=""https://www.sciencedail...",TRUE
3,1872.0,Is air pollution linked to greater risk of dem...,"<a href=""https://www.theguardian.com/environme...",TRUE
4,1956.0,Can eccentric exercises cause human hyperplasia?,With concentric regular exercises muscle hyper...,unproven


In [4]:
dataframe['rating'] = dataframe['rating'].replace('mixture', 'TRUE')
dataframe['rating'] = dataframe['rating'].replace('unproven', 'TRUE')
dataframe['rating'] = dataframe['rating'].replace('mostly-true', 'TRUE')
dataframe['rating'] = dataframe['rating'].replace('mostly-false', 'FALSE')

In [8]:
dataframe.head()

,id,statement,description,rating
0,6771.0,Dawn dish soap contains ammonia even though it...,Could Mixing Dawn Dish Soap with Clorox Bleach...,TRUE
1,2468.0,Is blue light harmful to our eyes?,It seems reasonable to reduce exposure to blue...,TRUE
2,1929.0,Are the 'Winter Blues' real?,"I've recently <a href=""https://www.sciencedail...",TRUE
3,1872.0,Is air pollution linked to greater risk of dem...,"<a href=""https://www.theguardian.com/environme...",TRUE
4,1956.0,Can eccentric exercises cause human hyperplasia?,With concentric regular exercises muscle hyper...,TRUE


In [4]:
dataframe.rating.value_counts()

rating
FALSE    353
TRUE     330
Name: count, dtype: int64

In [16]:
x = dataframe['statement']
y = dataframe['rating']

In [17]:
x

0      Dawn dish soap contains ammonia even though it...
1                     Is blue light harmful to our eyes?
2                           Are the 'Winter Blues' real?
3      Is air pollution linked to greater risk of dem...
4       Can eccentric exercises cause human hyperplasia?
                             ...                        
678               (noni|morinda citrifolia) treat cancer
679    (nonotus obliquus|chaga|Boletus obliquus|Polyp...
680    (orgone|orgonite|cloudbuster|Wilhelm Reich) tr...
681    (oxygen therapy|hyperbaric|hbot|O2 therapy|sup...
682    (Therapeutic touch|non-contact therapeutic tou...
Name: statement, Length: 683, dtype: object

In [18]:
y

0       TRUE
1       TRUE
2       TRUE
3       TRUE
4       TRUE
       ...  
678    FALSE
679    FALSE
680    FALSE
681    FALSE
682    FALSE
Name: rating, Length: 683, dtype: object

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [20]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0)
y_train

312    FALSE
202     TRUE
263     TRUE
395    FALSE
101     TRUE
       ...  
9       TRUE
359     TRUE
192    FALSE
629    FALSE
559    FALSE
Name: rating, Length: 546, dtype: object

In [21]:
y_train

312    FALSE
202     TRUE
263     TRUE
395    FALSE
101     TRUE
       ...  
9       TRUE
359     TRUE
192    FALSE
629    FALSE
559    FALSE
Name: rating, Length: 546, dtype: object

In [22]:
tfvect = TfidfVectorizer(stop_words='english',max_df=0.7)
tfid_x_train = tfvect.fit_transform(x_train)
tfid_x_test = tfvect.transform(x_test)

* max_df = 0.50 means "ignore terms that appear in more than 50% of the documents".
* max_df = 25 means "ignore terms that appear in more than 25 documents".

In [23]:
classifier = PassiveAggressiveClassifier(max_iter=50)
classifier.fit(tfid_x_train,y_train)

PassiveAggressiveClassifier(max_iter=50)

In [24]:
y_pred = classifier.predict(tfid_x_test)
score = accuracy_score(y_test,y_pred)
print(f'Accuracy: {round(score*100,2)}%')

Accuracy: 74.45%


In [26]:
cf = confusion_matrix(y_test,y_pred, labels=['TRUE','FALSE'])
print(cf)

[[55 18]
 [17 47]]


In [31]:
def misinfo_det(info):
    input_data = [info]
    vectorized_input_data = tfvect.transform(input_data)
    prediction = classifier.predict(vectorized_input_data)
    print(prediction)

In [32]:
misinfo_det('Is insomnia permanent?')

['FALSE']


In [33]:
import pickle
pickle.dump(classifier,open('newmodel.pkl', 'wb'))

In [34]:
# load the model from disk
loaded_model = pickle.load(open('newmodel.pkl', 'rb'))

In [35]:
def misinfo_det1(info):
    input_data = [info]
    vectorized_input_data = tfvect.transform(input_data)
    prediction = loaded_model.predict(vectorized_input_data)
    print(prediction)

In [37]:
misinfo_det1("Can diet impact sleep?")

['TRUE']


RNN Implementation

In [5]:
import tensorflow as tf
print(tf.__version__)


2.13.0-rc2


In [6]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from keras.models import Sequential
from sklearn.feature_extraction.text import CountVectorizer
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

In [7]:
num_of_categories = 1000
shuffled = dataframe.reindex(np.random.permutation(dataframe.index))
TRUE = shuffled[shuffled['rating'] == 'TRUE'][:num_of_categories]
FALSE = shuffled[shuffled['rating'] == 'FALSE'][:num_of_categories]
concated = pd.concat([TRUE,FALSE], ignore_index=True)
#Shuffle the dataset
concated = concated.reindex(np.random.permutation(concated.index))
concated['LABEL'] = 0

In [8]:
concated.loc[concated['rating'] == 'TRUE', 'LABEL'] = 0
concated.loc[concated['rating'] == 'FALSE', 'LABEL'] = 1
print(concated['LABEL'][:10])
labels = to_categorical(concated['LABEL'], num_classes=2)
print(labels[:10])
if 'rating' in concated.keys():
    concated.drop(['rating'], axis=1)

482    1
471    1
199    0
423    1
118    0
290    0
125    0
509    1
23     0
521    1
Name: LABEL, dtype: int64
[[0. 1.]
 [0. 1.]
 [1. 0.]
 [0. 1.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [1. 0.]
 [0. 1.]]


In [9]:
n_most_common_words = 8000
max_len = 130
tokenizer = Tokenizer(num_words=n_most_common_words, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
tokenizer.fit_on_texts(concated['statement'].values)
sequences = tokenizer.texts_to_sequences(concated['statement'].values)
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

X = pad_sequences(sequences, maxlen=max_len)

Found 2548 unique tokens.


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X , labels, test_size=0.25, random_state=42)

In [11]:
epochs = 10
emb_dim = 128
batch_size = 256
labels[:2]

array([[0., 1.],
       [0., 1.]], dtype=float32)

In [12]:
print((X_train.shape, y_train.shape, X_test.shape, y_test.shape))

model = Sequential()
model.add(Embedding(n_most_common_words, emb_dim, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.7))
model.add(LSTM(64, dropout=0.7, recurrent_dropout=0.7))
model.add(Dense(2, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
print(model.summary())
history = model.fit(X_train, y_train, epochs=10, batch_size=50,validation_split=0.2,callbacks=[EarlyStopping(monitor='val_loss',patience=3, min_delta=0)])

((512, 130), (512, 2), (171, 130), (171, 2))
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 130, 128)          1024000   
                                                                 
 spatial_dropout1d (Spatial  (None, 130, 128)          0         
 Dropout1D)                                                      
                                                                 
 lstm (LSTM)                 (None, 64)                49408     
                                                                 
 dense (Dense)               (None, 2)                 130       
                                                                 
Total params: 1073538 (4.10 MB)
Trainable params: 1073538 (4.10 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None
Epoch 1/10
9/9 [==================

In [13]:
accr = model.evaluate(X_test,y_test)
print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0],accr[1]))

6/6 [==============================] - 0s 16ms/step - loss: 0.4358 - acc: 0.8070
Test set
  Loss: 0.436
  Accuracy: 0.807


In [16]:
from keras.models import Sequential
from keras.layers import Embedding, SpatialDropout1D, LSTM, Dense
from keras.utils import plot_model

# Define the model architecture
model = Sequential()
model.add(Embedding(n_most_common_words, emb_dim, input_length=max_len))
model.add(SpatialDropout1D(0.7))
model.add(LSTM(64, dropout=0.7, recurrent_dropout=0.7))
model.add(Dense(2, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])

# Generate the neural network diagram
plot_model(model, to_file='neural_network_diagram.png', show_shapes=True)



You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [16]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import confusion_matrix
# Evaluation Metrics
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

precision, recall, f1_score, support = precision_recall_fscore_support(y_test_classes, y_pred_classes, average='weighted')
print("Precision: {:.2f}".format(precision))
print("Recall: {:.2f}".format(recall))
print("F1 Score: {:.2f}".format(f1_score))
print("Support: {}".format(support))

6/6 [==============================] - 0s 15ms/step
Precision: 0.89
Recall: 0.88
F1 Score: 0.88
Support: None


In [20]:
y_pred

array([[0.07344146, 0.92655855],
       [0.9593414 , 0.04065856],
       [0.19852361, 0.8014764 ],
       [0.9435826 , 0.05641738],
       [0.8380105 , 0.16198957],
       [0.72325367, 0.2767463 ],
       [0.62320924, 0.37679076],
       [0.1251723 , 0.8748277 ],
       [0.17144302, 0.828557  ],
       [0.17436695, 0.825633  ],
       [0.8336175 , 0.16638248],
       [0.15180837, 0.84819156],
       [0.87199533, 0.12800466],
       [0.12841009, 0.8715899 ],
       [0.29039648, 0.70960355],
       [0.9050143 , 0.09498571],
       [0.89465314, 0.10534686],
       [0.89408743, 0.10591257],
       [0.6688567 , 0.33114326],
       [0.68036073, 0.31963924],
       [0.09888972, 0.9011103 ],
       [0.10218637, 0.89781356],
       [0.7715207 , 0.2284793 ],
       [0.9116031 , 0.08839689],
       [0.38000423, 0.6199958 ],
       [0.12387266, 0.8761273 ],
       [0.29163417, 0.70836586],
       [0.84735656, 0.15264344],
       [0.94148546, 0.05851451],
       [0.30576628, 0.6942337 ],
       [0.

In [45]:
y_test

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [0., 1.

In [42]:
txt = ["Does taking apple cider vinegar as a supplement have any real health benefits?"]
seq = tokenizer.texts_to_sequences(txt)
padded = pad_sequences(seq, maxlen=max_len)
pred = model.predict(padded)
labels = ['TRUE', 'FALSE']
print(labels[np.argmax(pred)])

1/1 [==============================] - 0s 221ms/step
TRUE


In [113]:
y_pred_probabilities = model.predict(X_test)
y_pred = np.argmax(y_pred_probabilities, axis=1).astype("int64")
labels = to_categorical(concated['LABEL'], num_classes=2)
labels[labels == [1.,0.]]= [0]
labels[labels == [0.,1.]]= [1]
y_true = labels


6/6 [==============================] - 0s 13ms/step


In [114]:
y_true

array([[1., 0.],
       [1., 0.],
       [1., 1.],
       ...,
       [1., 0.],
       [1., 0.],
       [1., 0.]], dtype=float32)

In [100]:
y_pred

array([0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0,
       0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0], dtype=int64)

In [14]:
def rnn_det(txt):
    input_data = [txt]
    seq = tokenizer.texts_to_sequences(input_data)
    padded = pad_sequences(seq)
    prediction = model.predict(padded)
    labels = ['TRUE', 'FALSE']
    print(labels[np.argmax(prediction)])

In [15]:
rnn_det("Antibodies for the common cold produce a positive COVID-19 test; false-positive results from COVID-19 antibody testing are behind the COVID-19 cases reported in the U.S.")

1/1 [==============================] - 0s 236ms/step
FALSE


In [88]:
# imports
from tensorflow.keras.models import model_from_json


model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("model.h5")
print("Saved model to disk")



Saved model to disk
